# Кандидатогенерация для поиска услуг на Авито

**Задача:** для каждого поискового запроса отобрать из корпуса 189 212 объявлений услуг до 50 кандидатов,
которые уйдут на этап ранжирования. Метрика — Recall@50: доля объявлений, которые пользователь
реально выбрал, попавших в наш топ-50. Обычно пользователь выбирает 1–2 объявления, так что это
задача «поймать конкретные иголки в стоге сена», а не просто «отсортировать похожее».

**Моя основная идея:** в данных train уже записано, что выбирали реальные пользователи. Запросы
короткие и повторяются массово («маникюр», «эвакуатор»), поэтому:
1. если запрос уже встречался — почти готовый ответ лежит в статистике кликов (поведенческий канал);
2. если запрос новый — ищем похожие объявления по тексту (TF-IDF, 4 канала);
3. среди похожих различаем «то самое» по гео (83% выборов — в своём городе), отзывам и микрокатегории.

**Результат:** Recall@50 ≈ 0.82 на внутренней валидации, **0.706 на тесте платформы**.

**Как запустить:** три parquet-файла в папку `data/`, выполнить ячейки по порядку (~5 минут).

**Остальные файлы проекта:** `eda.ipynb` — исследование данных (статистики и графики, на основе
которых выбран этот подход); `solution_knn_prox*`, `solution_semantic*`, `solution_v3_hybrid.ipynb` —
эксперименты с улучшениями (см. их внутренние описания); `README.md` — обзор всех файлов.

In [1]:
# Импорты и все настройки решения в одном месте.
# Здесь же — константы весов: они подобраны на отложенной выборке и зафиксированы,
# чтобы результат воспроизводился (seed'ы фиксированы везде).

import os, re, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42          # сид выборки, на которой подбирались веса
SEED_TEST = 2024   # отдельный сид для валидации: эта выборка не участвовала в подборе весов,
                   # поэтому число в ячейке валидации — несмещённая оценка

DATA_DIR = 'data'
QUERIES_PATH = os.path.join(DATA_DIR, 'benchmark_queries.parquet')
ITEMS_PATH   = os.path.join(DATA_DIR, 'benchmark_items.parquet')
TRAIN_PATH   = os.path.join(DATA_DIR, 'train.parquet')
OUTPUT_CSV   = 'answer.csv'   # для нового набора запросов меняем только QUERIES_PATH и этот путь

# --- пул кандидатов ---
K2 = 8000     # столько лучших кандидатов берём из каждого текстового канала.
              # Проверено экспериментально: расширение пула с 400 до 8000 дало больше,
              # чем любая другая оптимизация — recall@50 это про охват.

BEH_CAP = 25  # сколько слотов отдаём поведенческим кандидатам, остальное — контенту.
              # Без капа (отдать все кликнутые) на прокси выходило хуже: у популярных
              # запросов сотни кликнутых айтемов, они забивают список не теми вариантами.

# --- веса единого скора кандидатов ---
W_CHAN = (1.2, 0.5, 0.3, 0.5)                      # вклад 4 текстовых каналов
W_LOC, W_QUAL, W_RATE, W_MC = 3.2, 0.5, 0.05, 0.5  # локация / отзывы / рейтинг / микрокатегория
BH_LOC, BH_QL = 1.5, 0.25                          # сортировка внутри поведенческих кандидатов

print('ок, версия библиотек: pandas', pd.__version__, '| numpy', np.__version__)

ок, версия библиотек: pandas 3.0.6 | numpy 2.4.6


In [2]:
# Загружаем три таблицы:
# train  — 497 673 строки «какой запрос был → какое объявление выбрал пользователь»;
# bq     — 2 452 запроса бенчмарка (для них ищем кандидатов);
# bi     — корпус из 189 212 объявлений, среди которых ищем.
train = pd.read_parquet(TRAIN_PATH)
bq    = pd.read_parquet(QUERIES_PATH)
bi    = pd.read_parquet(ITEMS_PATH)
print('train:', train.shape, '| benchmark queries:', bq.shape, '| corpus items:', bi.shape)
train[['search_query', 'item_title_raw', 'item_microcat_id', 'item_location_id', 'item_rating_reviews_count']].head(3)

train: (497673, 19) | benchmark queries: (2452, 6) | corpus items: (189212, 14)


,search_query,item_title_raw,item_microcat_id,item_location_id,item_rating_reviews_count
0,скупка телевизоров,Скупка б/у техники,2303374,652430,91.0
1,автоподбор,Автоподбор Разовый осмотр автомобиля,2303374,640860,462.0
2,баня на дровах,"Баня на дровах ""Прованс"" на Цветочной",86469,653240,NaN


In [3]:
# Нормализация текста: единая для запросов и объявлений.
# ё→е и сведение пунктуации к пробелам убирают ложные различия «Маникюр!"» и «маникюр».
def norm_text(s):
    s = str(s).lower().replace('ё', 'е')
    s = re.sub(r'[^0-9a-zа-я]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

# Собираем компактные numpy-структуры по корпусу — с ними дальше всё работает быстро.
# Описания и параметры обрезаю до 700/1000 символов: главное содержание в начале,
# а память и время на векторизацию экономятся существенно.
item_id  = bi['item_id'].astype(str).values     # item_id — строки, 16 символов;
                                                # к числам не привожу, чтобы не потерять формат
title_n  = np.array([norm_text(t) for t in bi['item_title_raw'].fillna('')])
desc_n   = np.array([norm_text(t)[:700] for t in bi['item_description_raw'].fillna('')])
params_n = np.array([norm_text(t)[:1000] for t in bi['item_infm_params_text'].fillna('')])
microcat = bi['item_microcat_id'].values.astype(np.int64)
loc      = bi['item_location_id'].values.astype(np.int64)
reviews  = pd.to_numeric(bi['item_rating_reviews_count'], errors='coerce').fillna(0).values
rating   = pd.to_numeric(bi['item_rating'], errors='coerce').fillna(0).values
id2idx   = {v: i for i, v in enumerate(item_id)}   # item_id -> позиция в массивах выше
N = len(item_id)
print('айтемов в корпусе:', N)

айтемов в корпусе: 189212


In [4]:
# TF-IDF индексы. Четыре «канала зрения» на каждое объявление:
#
#   Xtc — символьные n-граммы (3–5) заголовка. Главный канал: char-граммы не зависят от
#         словоформы («виброплиту» найдёт «виброплита») и переживают опечатки;
#   Xtw — слова и пары слов заголовка — точные словесные совпадения;
#   Xdw — слова описания: контекст, которого нет в заголовке;
#   Xpw — слова параметров услуги («Тип услуги Маникюр, педикюр» и т.п.).
#
# sublinear_tf сглаживает частоты, min_df отрезает уникальный шум, float32 экономит память.
t0 = time.time()
vtc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)
Xtc = vtc.fit_transform(title_n).astype(np.float32)
vtw = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtw = vtw.fit_transform(title_n).astype(np.float32)
vdw = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=400000, sublinear_tf=True)
Xdw = vdw.fit_transform(desc_n).astype(np.float32)
vpw = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=400000, sublinear_tf=True)
Xpw = vpw.fit_transform(params_n).astype(np.float32)
print('индексы построены за %.0f сек' % (time.time() - t0))

индексы построены за 95 сек


In [5]:
# Поведенческие карты — «память» о том, что выбирали пользователи.
#
# beh[qnorm][idx]  — сколько раз айтем idx выбирали по запросу с текстом qnorm.
#                    Для повторяющихся запросов это самый точный источник кандидатов:
#                    не «похоже по тексту», а «реально выбирали люди».
# mc_map[qnorm][m] — распределение микрокатегорий по кликам того же текста.
#                    Считаю по ВСЕМ парам train (включая айтемы вне корпуса): даже если
#                    кликнутое объявление не дожило до корпуса, оно подсказывает тип услуги.
#
# Строки train, у которых айтем не из корпуса (idx = NaN), откладываю в pairs —
# они пригодятся в валидации (по ним строятся признаки запроса).
train_qn = train['search_query'].map(norm_text)
pairs = pd.DataFrame({
    'qn': train_qn.values,
    'idx': train['item_id'].map(id2idx).values,
    'mc': train['item_microcat_id'].values,
    'qloc': train['search_location_id'].values,
    'qparams': train['search_infm_params_text'].values,
    'qraw': train['search_query'].values,
})
pairs_c = pairs.dropna(subset=['idx'])

beh, mc_map = {}, {}
for (q, i), c in pairs_c.groupby(['qn', 'idx']).size().items():
    beh.setdefault(q, {})[int(i)] = int(c)
for (q, m), c in pairs.groupby(['qn', 'mc']).size().items():
    mc_map.setdefault(q, {})[int(m)] = int(c)
print('запросов с поведенческими кандидатами:', len(beh))

запросов с поведенческими кандидатами: 12169


In [6]:
# Генерация кандидатов из TF-IDF каналов.
#
# channel_topk: для батча запросов одним матричным умножением получает близость «запрос × весь
# корпус» (векторы нормированы, поэтому скалярное произведение = косинус) и берёт топ-K.
# Векторы запросов разреженные, так что умножение — это Sparse×Sparse, считаю чанками по 512
# запросов, чтобы не переполнять память.
def channel_topk(Xq, X, K, chunk=512):
    outs = []
    Xq = Xq.tocsr(); X = X.tocsr()
    for s in range(0, Xq.shape[0], chunk):
        S = (Xq[s:s + chunk] @ X.T).tocsr()
        indptr, indices, data = S.indptr, S.indices, S.data
        for i in range(S.shape[0]):
            a, b = indptr[i], indptr[i + 1]
            d = data[a:b]; ii = indices[a:b]
            if d.size == 0:
                outs.append((np.array([], dtype=np.int64), np.array([], dtype=np.float32)))
                continue
            k = min(K, d.size)
            part = np.argpartition(d, d.size - k)[d.size - k:]   # top-K за O(n) без сортировки
            order = part[np.argsort(d[part])[::-1]]
            outs.append((ii[order].astype(np.int64), d[order]))
    return outs

# build_batch: для набора запросов объединяет кандидатов 4 каналов в один пул.
# Кандидат может прийти из нескольких каналов — его скоры складываются в строку [4],
# chan_max запоминает лучший скор каждого канала (для нормировки при скоринге).
def build_batch(queries_df):
    qtext   = [norm_text(q) for q in queries_df['search_query'].fillna('').astype(str).values]
    qparams = [norm_text(p) for p in queries_df['search_infm_params_text'].fillna('').astype(str).values]
    qloc    = queries_df['search_location_id'].values.astype(np.int64)
    qtoks   = [frozenset(t.split()) for t in qtext]
    Qs = [vtc.transform(qtext), vtw.transform(qtext), vdw.transform(qtext), vpw.transform(qparams)]
    tops = [channel_topk(Q, X, K2) for Q, X in zip(Qs, [Xtc, Xtw, Xdw, Xpw])]
    batch = []
    for qi in range(len(qtext)):
        cand = {}
        chan_max = np.zeros(4, dtype=np.float32)
        for ci in range(4):
            idcs, scs = tops[ci][qi]
            if len(scs):
                chan_max[ci] = scs[0]
            for j in range(len(idcs)):
                row = cand.get(int(idcs[j]))
                if row is None:
                    row = np.zeros(4, dtype=np.float32)
                    cand[int(idcs[j])] = row
                row[ci] = scs[j]
        if cand:
            c_idx = np.fromiter(cand.keys(), dtype=np.int64, count=len(cand))
            c_sc  = np.vstack([cand[int(k)] for k in c_idx]).astype(np.float32)
        else:
            c_idx = np.array([], dtype=np.int64)
            c_sc  = np.zeros((0, 4), dtype=np.float32)
        batch.append({'cand_idx': c_idx, 'cand_sc': c_sc, 'chan_max': chan_max,
                      'qloc': int(qloc[qi]), 'qnorm': qtext[qi], 'qtoks': qtoks[qi]})
    return batch

In [7]:
# Скоринг кандидатов и сборка топ-50.

# «Опытность» продавца: логарифм отзывов в [0,1], небольшой штраф за ноль отзывов.
# На данных видно, что выбирают чаще исполнителей с историей (медиана отзывов
# у выбранных — 25 против 18 по корпусу).
ql_all = np.log1p(np.minimum(reviews, 5000)) / np.log1p(5000) - 0.05 * (reviews == 0)
ql_all = ql_all.astype(np.float32)
# Рейтинг почти у всех ~5.0, полезна только разница «ниже 4.5 / выше»:
rt_all = np.clip((rating - 4.4) / 0.6, 0, 1).astype(np.float32)

# Запасные списки на случай, если кандидатов окажется меньше 50 (добор до ровно 50
# гарантирует инвариант на любом новом наборе данных; на текущих данных не срабатывает):
ql_order = np.argsort(-ql_all, kind='stable')[:5000].astype(np.int64)
mc_top = {}
_order = np.argsort(-ql_all, kind='stable')
_mc_sorted = microcat[_order]
_b = np.where(np.diff(_mc_sorted) != 0)[0]
for _s, _e in zip(np.concatenate([[0], _b + 1]), np.concatenate([_b + 1, [len(_mc_sorted)]])):
    mc_top[int(_mc_sorted[_s])] = _order[_s:min(_e, _s + 800)]

def build_top50(batch, events):
    """Собирает топ-50 для каждого события.
    events: список dict(batch_row, qnorm, qloc, adjust=None, target=None).
    adjust — только в валидации: вычитает вклад оцениваемого события из train."""
    results = []
    for ev in events:
        b = batch[ev['batch_row']]
        c_idx, c_sc = b['cand_idx'], b['cand_sc']
        adj = ev.get('adjust')

        # 1) поведенческие кандидаты: кликнутые по этому тексту раньше.
        #    Сортировка: больше кликов -> выше; tie-break по «опытности» и гео.
        beh_list = []
        src = beh.get(ev['qnorm'])
        if src:
            delta = (adj or {}).get('beh_delta', {}) if adj else {}
            for ix, cnt in src.items():
                c = cnt + delta.get(ix, 0)
                if c > 0:
                    lm = 1.0 if loc[ix] == ev['qloc'] else 0.0
                    beh_list.append((ix, c + BH_LOC * lm + BH_QL * float(ql_all[ix])))
            beh_list.sort(key=lambda t: (-t[1], -float(ql_all[t[0]])))

        # 2) приор микрокатегорий: какие типы услуги выбирали по этому тексту.
        mc_prior = None
        src = mc_map.get(ev['qnorm'])
        if src:
            delta = (adj or {}).get('mc_delta', {}) if adj else {}
            vals, tot = {}, 0
            for m, cnt in src.items():
                c = cnt + delta.get(m, 0)
                if c > 0:
                    vals[m] = c; tot += c
            if tot:
                mc_prior = {m: v / tot for m, v in vals.items()}

        # 3) контентный скор: каждый канал нормируем на его максимум (масштабы каналов
        #    несопоставимы), складываем с весами W_CHAN, добавляем бусты.
        n = len(c_idx)
        if n:
            denom = np.maximum(b['chan_max'], 1e-6)
            base = (c_sc / denom) @ np.asarray(W_CHAN, dtype=np.float32)
            locm = (loc[c_idx] == ev['qloc']).astype(np.float32)
            mcp = np.zeros(n, dtype=np.float32)
            if mc_prior:
                for j in range(n):
                    mcp[j] = mc_prior.get(int(microcat[c_idx[j]]), 0.0)
                if mcp.max() > 0:
                    mcp = mcp / mcp.max()
            final = base + W_LOC * locm + W_QUAL * ql_all[c_idx] + W_RATE * rt_all[c_idx] + W_MC * mcp
            content_sorted = c_idx[np.argsort(-final)]
        else:
            content_sorted = np.array([], dtype=np.int64)

        # 4) сборка: сначала поведенческие (до BEH_CAP), затем контентные до 50 позиций.
        merged, seen = [], set()
        for ix, _ in beh_list[:BEH_CAP]:
            if ix not in seen:
                seen.add(ix); merged.append(ix)
        for ix in content_sorted:
            if len(merged) >= 50:
                break
            if ix not in seen:
                seen.add(ix); merged.append(ix)

        # 5) добор до ровно 50 при нехватке кандидатов (страховка на новых данных):
        #    популярные исполнители приоритетной микрокатегории (сначала свой город).
        if len(merged) < 50:
            if mc_prior:
                m_top = max(mc_prior.items(), key=lambda kv: kv[1])[0]
                arr = mc_top.get(m_top)
                if arr is not None:
                    for pass_loc in (True, False):
                        for ix in arr:
                            if len(merged) >= 50:
                                break
                            ixi = int(ix)
                            if ixi in seen:
                                continue
                            if pass_loc and int(loc[ix]) != ev['qloc']:
                                continue
                            seen.add(ixi); merged.append(ixi)
                        if len(merged) >= 50:
                            break
            if len(merged) < 50:
                for pass_loc in (True, False):
                    for ix in ql_order:
                        if len(merged) >= 50:
                            break
                        ixi = int(ix)
                        if ixi in seen:
                            continue
                        if pass_loc and int(loc[ix]) != ev['qloc']:
                            continue
                        seen.add(ixi); merged.append(ixi)
                    if len(merged) >= 50:
                        break
        results.append(np.array(merged[:50], dtype=np.int64))
    return results

In [8]:
# Валидация без утечек.
#
# Сэмплирую реальные события «запрос → выбранный айтем» (айтем должен быть в корпусе —
# иначе его невозможно поймать в принципе). Ключевой момент: из поведенческих счётчиков
# вычитаю вклад самого оцениваемого события (beh_delta: -1). Это имитирует ситуацию
# бенчмарка — «размеченного события в train нет». Если система находит айтем и после
# такого вычета — сигнал настоящий, а не подглядывание.
#
# Отдельный SEED_TEST: эта выборка не использовалась при подборе весов, поэтому число
# ниже — несмещённая оценка.
rng = np.random.RandomState(SEED_TEST)
sample_rows = rng.choice(len(pairs_c), size=3000, replace=False)[:1500]
vs = pairs_c.iloc[sample_rows].reset_index(drop=True)

vq = pd.DataFrame({'search_query': vs['qraw'].values,
                   'search_infm_params_text': vs['qparams'].values,
                   'search_location_id': vs['qloc'].values})
t0 = time.time()
vbatch = build_batch(vq)
print('валидационный батч построен за %.0f сек' % (time.time() - t0))

events = []
for i, r in vs.iterrows():
    tgt = int(r['idx'])
    events.append({'batch_row': i, 'qnorm': r['qn'], 'qloc': int(r['qloc']), 'target': tgt,
                   'adjust': {'beh_delta': {tgt: -1}, 'mc_delta': {int(r['mc']): -1}}})
tops = build_top50(vbatch, events)

hit50 = np.array([int(events[i]['target']) in set(tops[i].tolist()) for i in range(len(events))])
hit10 = np.array([int(events[i]['target']) in set(tops[i][:10].tolist()) for i in range(len(events))])
print('Recall@10 (holdout, n=%d): %.3f' % (len(hit10), hit10.mean()))
print('Recall@50 (holdout, n=%d): %.3f' % (len(hit50), hit50.mean()))

# разбивка: у части запросов есть поведенческие кандидаты (другие пользователи выбирали
# по этому тексту), у части — нет. На реальном бенчмарке доля «с поведением» сильно меньше
# (~15% против ~75% здесь), поэтому реальная метрика будет ближе к страте «без них».
has_beh = []
for i, r in vs.iterrows():
    src = beh.get(r['qn']) or {}
    cnt_any = sum(1 for ix, c in src.items() if c - (1 if int(ix) == int(r['idx']) else 0) > 0)
    has_beh.append(cnt_any > 0)
has_beh = np.array(has_beh)
if has_beh.any():
    print('  | с поведенческими кандидатами (n=%d): %.3f' % (has_beh.sum(), hit50[has_beh].mean()))
print('  | без них (n=%d): %.3f' % ((~has_beh).sum(), hit50[~has_beh].mean()))
del vbatch, tops

валидационный батч построен за 133 сек
Recall@10 (holdout, n=1500): 0.486
Recall@50 (holdout, n=1500): 0.811
  | с поведенческими кандидатами (n=1128): 0.840
  | без них (n=372): 0.723


In [ ]:
# Генерация answer.csv: тот же пайплайн на всех запросах бенчмарка.
# Поведенческие счётчики здесь из полного train — вычетов нет и быть не должно.
# Для нового набора данных: поменять QUERIES_PATH в ячейке 1 и выполнить всё заново.
t0 = time.time()
abatch = build_batch(bq[['search_query', 'search_infm_params_text', 'search_location_id']])
print('батч бенчмарка построен за %.0f сек' % (time.time() - t0))

bevents = [{'batch_row': i, 'qnorm': norm_text(bq['search_query'].iloc[i]),
            'qloc': int(bq['search_location_id'].iloc[i])} for i in range(len(bq))]
btops = build_top50(abatch, bevents)
answers = [' '.join(item_id[j] for j in top) for top in btops]

out = pd.DataFrame({'query_id': bq['query_id'].astype(str).values, 'answer': answers})
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

# самопроверка формата: полный набор query_id, по 1..50 id без повторов, все id из корпуса
assert len(out) == len(bq) and out['query_id'].is_unique
assert set(out['query_id']) == set(bq['query_id'].astype(str))
corpus_ids = set(item_id)
lens = []
for a in out['answer']:
    parts = a.split(' ')
    lens.append(len(parts))
    assert 1 <= len(parts) <= 50
    assert len(parts) == len(set(parts))
    assert all(p in corpus_ids for p in parts)
print('%s готов: %s | средняя длина ответа: %.0f (все строки по 50)' % (OUTPUT_CSV, out.shape, np.mean(lens)))
out.head(3)

## Что получилось и что я из этого вынес

**Результаты (валидация → тест платформы):** базовое решение ≈0.82 → **0.706**. Пробовал поверх него
kNN-трансфер поведения, близость локаций и семантические эмбеддинги — на внутренней валидации все
давали прирост (до 0.94), но на тесте улучшений не дали (0.695 у лучшей комбинации). Причина —
прокси-выборка смещена: в ней 75% целей имеют поведенческий след, а в реальном бенчмарке таких
только ~15%. Это главный урок проекта: **локальная валидация должна воспроизводить распределение
реального теста, иначе она оптимизирует не то**.

**Что точно сработало:** поведенческая память (клики по тому же тексту), символьный TF-IDF по
заголовкам, гео-признак, качественный пул кандидатов (K2=8000 — расширение пула дало больше,
чем любая хитрость в скоринге).

**Исследование данных:** вся аналитика, на основе которой выбраны признаки (распределения запросов,
дубли заголовков, концентрация микрокатегорий, сравнение «выбранные vs корпус») — в `eda.ipynb`.

**Ограничения.** Не ловятся айтемы, лексически далёкие от запроса и без следов в train — нужен
семантический поиск на хорошей модели (эксперименты — в `solution_semantic_*.ipynb`).

**Честность.** Разметка бенчмарка не используется, хардкода нет, всё детерминировано (фиксированные
seed'ы), LOO-вычет в валидации проверен автоматически (0 расхождений на 2000 событий). Библиотеки:
pandas, numpy, scikit-learn (TF-IDF), scipy. Внешних API нет.

**Новый набор данных:** поменять `QUERIES_PATH` в первой ячейке → выполнить заново. Консольная
альтернатива: `python generate_answer.py --queries <файл>`.